```mermaid

classDiagram
    direction TB

    class Element {
        +str name
        +str symbol
        +int atomic_number
        +float atomic_mass
        +__repr__()
    }

    class Atom {
        +Element element
        +symbol
        +name
        +atomic_number
        +mass
        +__repr__()
    }

    class Bond {
        +Atom atom1
        +Atom atom2
        +str bond_type
        +order
        +contains(atom)
        +other(atom)
        +__repr__()
    }

    class Molecule {
        +list atoms
        +list bonds
        +mass
        +formula
        -_validate()
        +neighbors(atom)
        +degree(atom)
        +describe()
        +__repr__()
    }

    Element "1" <-- "0..*" Atom : describes

    Molecule "1" *-- "1..*" Atom : contains
    Molecule "1" *-- "0..*" Bond : contains

    Bond "0..*" --> "2" Atom : connects

```

In [1]:
from collections import Counter


# =========================================================
# ELEMENT
# =========================================================

class Element:
    """
    Describes a chemical element.

    An Element is a definition/type rather than an individual atom.
    """

    def __init__(self, name, symbol, atomic_number, atomic_mass):
        self.name = name
        self.symbol = symbol
        self.atomic_number = atomic_number
        self.atomic_mass = atomic_mass

    def __repr__(self):
        return self.symbol


# =========================================================
# ATOM
# =========================================================

class Atom:
    """
    Represents an individual atom belonging to an Element.
    """

    def __init__(self, element):
        self.element = element

    @property
    def name(self):
        return self.element.name

    @property
    def symbol(self):
        return self.element.symbol

    @property
    def atomic_number(self):
        return self.element.atomic_number

    @property
    def mass(self):
        return self.element.atomic_mass

    def __repr__(self):
        return self.symbol


# =========================================================
# BOND
# =========================================================

class Bond:
    """
    Represents a bond connecting two Atom objects.
    """

    BOND_SYMBOLS = {
        "single": "-",
        "double": "=",
        "triple": "≡"
    }

    def __init__(self, atom1, atom2, bond_type="single"):

        if not isinstance(atom1, Atom):
            raise TypeError("atom1 must be an Atom.")

        if not isinstance(atom2, Atom):
            raise TypeError("atom2 must be an Atom.")

        if atom1 is atom2:
            raise ValueError("An atom cannot be bonded to itself.")

        if bond_type not in self.BOND_SYMBOLS:
            raise ValueError(
                "bond_type must be 'single', 'double', or 'triple'."
            )

        self.atom1 = atom1
        self.atom2 = atom2
        self.bond_type = bond_type

    @property
    def order(self):

        orders = {
            "single": 1,
            "double": 2,
            "triple": 3
        }

        return orders[self.bond_type]

    def contains(self, atom):
        return atom is self.atom1 or atom is self.atom2

    def other(self, atom):

        if atom is self.atom1:
            return self.atom2

        if atom is self.atom2:
            return self.atom1

        raise ValueError("The atom is not part of this bond.")

    def __repr__(self):

        connector = self.BOND_SYMBOLS[self.bond_type]

        return f"{self.atom1}{connector}{self.atom2}"


# =========================================================
# MOLECULE
# =========================================================

class Molecule:
    """
    Represents a molecule composed of atoms and bonds.
    """

    def __init__(self, atoms, bonds=None):

        self.atoms = list(atoms)
        self.bonds = list(bonds) if bonds is not None else []

        self._validate()

    # -----------------------------------------------------
    # Validation
    # -----------------------------------------------------

    def _validate(self):

        for atom in self.atoms:

            if not isinstance(atom, Atom):
                raise TypeError(
                    "Every item in atoms must be an Atom."
                )

        for bond in self.bonds:

            if not isinstance(bond, Bond):
                raise TypeError(
                    "Every item in bonds must be a Bond."
                )

            if (
                bond.atom1 not in self.atoms
                or bond.atom2 not in self.atoms
            ):
                raise ValueError(
                    "Bond contains an atom that is not "
                    "part of the molecule."
                )

    # -----------------------------------------------------
    # Molecular mass
    # -----------------------------------------------------

    @property
    def mass(self):

        return sum(
            atom.mass
            for atom in self.atoms
        )

    # -----------------------------------------------------
    # Formula
    # -----------------------------------------------------

    @property
    def formula(self):

        counts = Counter(
            atom.symbol
            for atom in self.atoms
        )

        formula = ""

        for symbol, count in counts.items():

            formula += symbol

            if count > 1:
                formula += str(count)

        return formula

    # -----------------------------------------------------
    # Graph operations
    # -----------------------------------------------------

    def neighbors(self, atom):

        if atom not in self.atoms:
            raise ValueError(
                "Atom does not belong to this molecule."
            )

        result = []

        for bond in self.bonds:

            if bond.contains(atom):
                result.append(
                    bond.other(atom)
                )

        return result

    def degree(self, atom):

        return len(
            self.neighbors(atom)
        )

    # -----------------------------------------------------
    # Display information
    # -----------------------------------------------------

    def describe(self):

        print(f"Molecule: {self.formula}")
        print(f"Mass: {self.mass:.3f}")
        print(f"Atoms: {self.atoms}")
        print(f"Bonds: {self.bonds}")

    def __repr__(self):

        return self.formula

In [18]:
hydrogen = Element(
    name="Hydrogen",
    symbol="H",
    atomic_number=1,
    atomic_mass=1.008
)

carbon = Element(
    name="Carbon",
    symbol="C",
    atomic_number=6,
    atomic_mass=12.011
)

nitrogen = Element(
    name="Nitrogen",
    symbol="N",
    atomic_number=7,
    atomic_mass=14.007
)

oxygen = Element(
    name="Oxygen",
    symbol="O",
    atomic_number=8,
    atomic_mass=15.999
)

sodium = Element(
    name="Sodium",
    symbol="Na",
    atomic_number=11,
    atomic_mass=22.99
)

chlorine = Element(
    name="Chlorine",
    symbol="Cl",
    atomic_number=17,
    atomic_mass=35.45
)

In [19]:
H1 = Atom(hydrogen)
H2 = Atom(hydrogen)
O1 = Atom(oxygen)

In [20]:
OH1 = Bond(O1, H1)
OH2 = Bond(O1, H2)

In [21]:
water = Molecule(
    atoms=[
        H1,
        H2,
        O1
    ],
    bonds=[
        OH1,
        OH2
    ]
)

In [22]:
print(water.formula)

H2O


In [23]:
Na1 = Atom(sodium)
Cl1 = Atom(chlorine)

NaCl = Bond(Na1, Cl1)

In [24]:
salt = Molecule(
    atoms = [
        Na1, Cl1
    ],
    bonds=[NaCl]
)

In [26]:
salt.formula

'NaCl'

In [28]:
NaCl

Na-Cl